In [8]:
import pandas as pd
import numpy as np

from scipy.stats.mstats import winsorize
from scipy import stats

import requests
from pandas import json_normalize

In [9]:
#load dataset
df = pd.read_csv('/content/housing_dirty.csv')
print(df.head())
print(df.info())
print(df.shape)


   id  luas_m2  harga_juta   kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  jogja    2.0          2000    baik
1   2    254.0       761.0  Medan    NaN          1995   Bagus
2   3    249.7       895.0  Depok    NaN          1983    baik
3   4     49.7       178.0    YGY    5.0          2013    baik
4   5    133.4       424.0  Medan    5.0          2004  Sedang
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
(130, 7)


In [10]:
#Eksplorasi Missing Values
print(df.isnull().sum())
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct)


id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64
id               0.000000
luas_m2         13.846154
harga_juta      13.076923
kota             0.000000
kamar            7.692308
tahun_bangun     0.000000
kondisi          0.000000
dtype: float64


In [11]:
#Cek Data Duplikat
jumlah_duplikat = df.duplicated().sum()

print("Jumlah duplikat:", jumlah_duplikat)
df[df.duplicated()]

Jumlah duplikat: 0


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi


In [12]:
#Hapus Data Duplikat
df.drop_duplicates(inplace=True)

print("Shape setelah hapus duplikat:", df.shape)


Shape setelah hapus duplikat: (130, 7)


In [13]:
#Normalisasi String
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print(df['kota'].unique())
print(df['kondisi'].unique())


['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']
['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']


In [14]:
#Imputasi Missing Values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
print(df.isnull().sum())

id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [15]:
#Deteksi Outlier dengan IQR
def deteksi_outlier_iqr(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier = df[(df[kolom] < lower) | (df[kolom] > upper)]

    return lower, upper, outlier

lower, upper, outlier_df = deteksi_outlier_iqr(df, 'harga_juta')

print("Batas bawah:", lower)
print("Batas atas:", upper)
print("Jumlah outlier:", len(outlier_df))

Batas bawah: -422.75
Batas atas: 1719.25
Jumlah outlier: 3


In [16]:
#Menangani Outlier dengan Capping
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)


In [17]:
#Validasi Dataset Bersih
print(df.isnull().sum().sum())
print(df.duplicated().sum())

0
0


In [18]:
#Simpan Dataset Bersih
df.to_csv('housing_clean.csv', index=False)
print("Dataset berhasil disimpan!")

Dataset berhasil disimpan!


In [19]:
#Akses REST API
URL = "https://jsonplaceholder.typicode.com/users"

response = requests.get(URL, timeout=10)

if response.status_code == 200:

    data = response.json()

    df_api = json_normalize(data)

    print(df_api.head())

else:
    print("Error:", response.status_code)

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh

In [20]:
#Menampilkan Kolom Tertentu dari API
print(df_api[['id', 'name', 'email']])

   id                      name                      email
0   1             Leanne Graham          Sincere@april.biz
1   2              Ervin Howell          Shanna@melissa.tv
2   3          Clementine Bauch         Nathan@yesenia.net
3   4          Patricia Lebsack  Julianne.OConner@kory.org
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me
8   9           Glenna Reichert    Chaim_McDermott@dana.io
9  10        Clementina DuBuque     Rey.Padberg@karina.biz
